In [ ]:
#IMDb Dataset
#   ↓
#Documents
##   ↓
#OpenAI Embeddings
#(text-embedding model)
#   ↓
#VectorStoreIndex
#   ↓
#Similarity Search
#   ↓
#OpenAI LLM Response
#(Movie Recommendation)

In [ ]:
!pip install pandas scikit-learn

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab_Notebooks/Interview/Projects/Project/proj10_moviereview/IMDb_Top_250_Movies.csv", encoding="latin1")

df.head()

,Sl_No,Name,Release_Year,Duration,Certificate,Rating,Votes,Director,Stars,Description
0,1,The Shawshank,1994,142,15,9.3,2765122,Frank Darabont,"Tim Robbins, Morgan Freeman, Bob Gunton","Over the course of several years, two convicts..."
1,2,The Godfather,1972,175,X,9.2,1924399,Francis Ford Coppola,"Marlon Brando, Al Pacino, James Caan","Don Vito Corleone, head of a mafia family, dec..."
2,3,The Dark Knight,2008,152,12A,9.0,2738387,Christopher Nolan,"Christian Bale, Heath Ledger, Aaron Eckhart",When the menace known as the Joker wreaks havo...
3,4,The Godfather Part II,1974,202,X,9.0,1309392,Francis Ford Coppola,"Al Pacino, Robert De Niro, Robert Duvall",The early life and career of Vito Corleone in ...
4,5,12 Angry Men,1957,96,U,9.0,819728,Sidney Lumet,"Henry Fonda, Lee J. Cobb, Martin Balsam",The jury in a New York City murder trial is fr...


In [ ]:
!pip install llama-index llama-index-embeddings-openai llama-index-llms-openai pandas

INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from llama_index.core import Document

documents = []

for _, row in df.iterrows():

    movie_text = f"""
    Movie Title: {row['Description']}
    Release Year: {row['Release_Year']}
    Director: {row['Director']}
    Rating: {row['Rating']}
    Description: {row['Description']}
    """

    documents.append(Document(text=movie_text))

In [ ]:
##Step 5 — Use OpenAI Embeddings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

llm = OpenAI(
    model="gpt-4o-mini"
)

In [ ]:
##Step 6 — Create OpenAI Vector Index
from llama_index.core import VectorStoreIndex, Settings

Settings.embed_model = embed_model
Settings.llm = llm

index = VectorStoreIndex.from_documents(documents)

In [ ]:
##Step 7 — Create RAG Query Engine
query_engine = index.as_query_engine(
    similarity_top_k=5
)

In [ ]:
#Step 8 — Movie Recommendation Query
query = """
Recommend movies similar to Interstellar.
Explain why they are similar.
Mention rating and director.
"""

response = query_engine.query(query)

print(response)

Here are some movies that are similar to "Interstellar":

1. **Inception**
   - **Director:** Christopher Nolan
   - **Rating:** 8.8
   - **Similarity:** Both films are directed by Christopher Nolan and explore complex themes involving time and reality. "Inception" delves into the manipulation of dreams, while "Interstellar" tackles space travel and the relativity of time.

2. **WALL-E**
   - **Director:** Andrew Stanton
   - **Rating:** 8.4
   - **Similarity:** This film features a futuristic setting and themes of humanity's survival, much like "Interstellar." It also emphasizes the importance of environmental stewardship and the consequences of neglecting our planet.

3. **Alien**
   - **Director:** Ridley Scott
   - **Rating:** 8.5
   - **Similarity:** Both films involve space exploration and the unknown dangers that come with it. "Alien" focuses on the crew's encounter with a deadly lifeform, paralleling the high-stakes scenarios faced by the characters in "Interstellar."

4. **200